## simple_triton example

This notebook illustrates how to encode tiles from a whole-slide image using a Triton inference server.

- See https://github.com/PathologyDataScience/simple_triton for details on launching the triton server container and mounting the model repository notebook
- Requires installation of [`mil`](https://github.com/PathologyDataScience/mil) and [`histomics_stream`](https://github.com/DigitalSlideArchive/HistomicsStream)
- Run this notebook in a container with `--network=host` so that it can reach the Triton container
- Mount your model repository directory to the triton container

In [ ]:
# install large_image with tile sources
!pip install histomics_stream 'large_image[tiff]' \
  scikit_image --find-links https://girder.github.io/large_image_wheels

# install simple_triton
!pip install ../../simple_triton

# install ray tune dependencies and mil
!pip install pooch
!pip install ../../mil

## Run client with no GPUs

If running Triton and the client on the same machine, we want to stop the client tensorflow from consuming GPU resources. By default, TensorFlow maps nearly all available GPU memory.

In [2]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf

assert len(tf.config.list_physical_devices("GPU")) == 0

## Download sample data

Download the hosted whole-slide image and mask.

In [3]:
import pooch

# download whole slide image and corresponding mask
wsi_path = pooch.retrieve(
    fname="TCGA-AN-A0G0-01Z-00-DX1.svs",
    url="https://drive.usercontent.google.com/download?id=19agE_0cWY582szhOVxp9h3kozRfB4CvV&export=download&confirm=t",
    known_hash="d046f952759ff6987374786768fc588740eef1e54e4e295a684f3bd356c8528f",
    path=str(pooch.os_cache("pooch")) + os.sep + "wsi",
)
mask_path = pooch.retrieve(
    fname="TCGA-AN-A0G0-01Z-00-DX1.mask.png",
    url="https://drive.usercontent.google.com/download?id=17GOOHbL8Bo3933rdIui82akr7stbRfta&export=download&confirm=t",
    known_hash="bb657ead9fd3b8284db6ecc1ca8a1efa57a0e9fd73d2ea63ce6053fbd3d65171",
    path=str(pooch.os_cache("pooch")) + os.sep + "wsi",
)

## Create an encoder model

`tf_encoder` creates encoder models from the available models in `tf.keras.applications`. Running this takes time as the model is downloaded. The encoder model is saved into the designated triton server model respository that is mounted within the triton server container.

In [4]:
import numpy as np
from pprint import pprint
from simple_triton.encoders import tf_encoder
from simple_triton.model import TritonModel

# model parameters
keras_name = "EfficientNetV2S"
model_name = f"{keras_name}.tensorflow"  # set model name
tile = 224

# create the model and capture output dimensionality
if not os.path.exists(os.path.join("~/models", model_name)):
    dimension_output = tf_encoder(
        "~/models", keras_name, model_name, input_shape=(tile, tile, 3), pooling="avg"
    )

## Load the model using `TritonModel`

After creating the model, we load the model into Triton using the `TritonModel` class. This class contains methods for loading, unloading, and checking the status of models. To load the model we create a simple configuration with batch size 64, and allow Triton to generate the remaining configuration fields. By default it will load a single copy of the model on each system GPU, and will often automatically set optimizations like pinned memory.

In [5]:
# triton parameters
url = "localhost:8001"  # url for grpc access to tirton server

# load tensorflow model - set maximum batch size
model = TritonModel(model_name, url)
model.load(config={"maxBatchSize": 64})
assert model.is_loaded()
pprint(model.get_config())

{'backend': 'tensorflow',
 'defaultModelFilename': 'model.savedmodel',
 'dynamicBatching': {'preferredBatchSize': [64]},
 'input': [{'dataType': 'TYPE_UINT8',
            'dims': ['224', '224', '3'],
            'name': 'input_0'}],
 'instanceGroup': [{'count': 1,
                    'gpus': [0, 1, 2, 3, 4, 5, 6],
                    'kind': 'KIND_GPU',
                    'name': 'EfficientNetV2S.tensorflow'}],
 'maxBatchSize': 64,
 'name': 'EfficientNetV2S.tensorflow',
 'optimization': {'inputPinnedMemory': {'enable': True},
                  'outputPinnedMemory': {'enable': True}},
 'output': [{'dataType': 'TYPE_FP32', 'dims': ['1280'], 'name': 'output_0'}],
 'platform': 'tensorflow_savedmodel',
 'versionPolicy': {'latest': {'numVersions': 1}}}


## Advanced configuration with `ConfigBuilder`

The `ConfigBuilder` class provides access to advanced configuration options like backend optimizations. Here, we create a duplicate model on each GPU (`count=2`) and convert the model to mixed precision to improve speed and memory usage. The model is reloaded using this advanced configuration.

In [6]:
from simple_triton.config import ConfigBuilder

# initialize builder with a basic configuration
builder = ConfigBuilder(model_name, config={"maxBatchSize": 64})

# increase the number of model instances per GPU to 2
builder.add_instance_group(count=2)

# add automatic mixed precision
builder.add_mixed_precision()

# re-load model with new config
model.load(config=builder.config)

# print config
pprint(model.get_config())

{'backend': 'tensorflow',
 'defaultModelFilename': 'model.savedmodel',
 'dynamicBatching': {'preferredBatchSize': [64]},
 'input': [{'dataType': 'TYPE_UINT8',
            'dims': ['224', '224', '3'],
            'name': 'input_0'}],
 'instanceGroup': [{'count': 2,
                    'gpus': [0, 1, 2, 3, 4, 5, 6],
                    'kind': 'KIND_GPU',
                    'name': 'EfficientNetV2S.tensorflow_0'}],
 'maxBatchSize': 64,
 'name': 'EfficientNetV2S.tensorflow',
 'optimization': {'executionAccelerators': {'gpuExecutionAccelerator': [{'name': 'auto_mixed_precision'}]},
                  'inputPinnedMemory': {'enable': True},
                  'outputPinnedMemory': {'enable': True}},
 'output': [{'dataType': 'TYPE_FP32', 'dims': ['1280'], 'name': 'output_0'}],
 'platform': 'tensorflow_savedmodel',
 'versionPolicy': {'latest': {'numVersions': 1}}}


## Run the inference

First, a histomics stream study is created defining the tiles that need to be read based on the whole-slide image, tissue mask, and desired magnification, tile size, and tile overlap. The chunk parameter is used to group tiles during disk reads to maximize throughput. This study initializes a `LargeimagePrefetch` iterator that generates batches of tiles and tile metadata using prefetching.

This iterator is passed to the inference function that is parameterized by the number of tiles per batch, the number of workers, and the maximum number of pending inferences per worker.

In [7]:
from simple_triton.feature_extraction import inference, study
from simple_triton.tile_iterators import TiffPrefetch
from simple_triton.utils import analyze
from time import time

# slide parameters
batch = 64
magnification = 20.0
chunk = 896
mask_threshold = 0.5

# create a histomics-stream study from a wsi/mask pair
hs_study = study(
    (wsi_path, mask_path),
    t=(tile, tile),
    chunk=(chunk, chunk),
    objective=magnification,
    mask_threshold=mask_threshold,
)

# tile iterator parameters
batch = 64
prefetch = 4
workers = 16  # total number of tile
icc = True  # apply ICC color correction

# inference parameters
limit = 1  # limit on number of pending requests per worker
verbose = True  # display inference statistics and debugging information

# start timer
start = time()

# create tile iterator
iterator = TiffPrefetch(hs_study, np.uint8, icc, batch, prefetch, workers)

# inference
features, metadata, times, failures = inference(
    iterator, model_name, url="localhost:8001", limit=limit, rest=0.0
)

# display elapsed time
print(f"Total elapsed time: {time()-start}")

# analyze performance
analyze(times)

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
2024-07-30 01:45:24,444	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2024-07-30 01:45:24,543	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2024-07-30 01:45:24,669	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Total elapsed time: 48.559579610824585
                        median    stdv    min    max
--------------------  --------  ------  -----  -----
total (sec)               0.14    1.07   0.12   3.18
read (sec)                0.01    0.00   0.00   0.02
submission (sec)          0.02    0.00   0.01   0.03
completion (sec)          0.10    1.07   0.09   3.15
retrieval (sec)           0.01    0.00   0.00   0.01
--------------------  --------  ------  -----  -----
read (% total)            4.35    2.31   0.09   9.16
submission (% total)     11.15    4.96   0.40  18.37
completion (% total)     78.74    9.00  69.48  99.24
retrieval (% total)       5.60    2.47   0.00   7.70


## Write features to .tfr

In [8]:
from mil.io.reader import read_record, peek
from mil.io.writer import write_record

# concatenate features
features = np.concatenate(features[0], axis=0)

# create dummy labels
labels = {"labels": np.random.uniform(size=(10))}

# write to tfrecord
write_record(
    "./triton.tfr", features, metadata, labels, structured=False, precision=tf.float16
)

# get list of .tfr variables for de-serialization
serialized = list(tf.data.TFRecordDataset(["./triton.tfr"]))[0]
variables = peek(serialized)

# verify reading
read_record(serialized, variables, structured=False, precision=tf.float16)

2024-07-30 01:46:45.310238: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


(<tf.Tensor: shape=(4658, 1280), dtype=float16, numpy=
 array([[ 0.007492 , -0.00413  ,  0.0142   , ..., -0.01697  , -0.0001816,
          0.015144 ],
        [ 0.007492 , -0.00413  ,  0.0142   , ..., -0.01697  , -0.0001816,
          0.015144 ],
        [ 0.007492 , -0.00413  ,  0.0142   , ..., -0.01697  , -0.0001816,
          0.015144 ],
        ...,
        [ 0.00754  , -0.004005 ,  0.014175 , ..., -0.01701  , -0.0001416,
          0.015274 ],
        [ 0.00754  , -0.004005 ,  0.014175 , ..., -0.01701  , -0.0001416,
          0.015274 ],
        [ 0.00754  , -0.004005 ,  0.014175 , ..., -0.01701  , -0.0001416,
          0.015274 ]], dtype=float16)>,
 {'labels': <tf.Tensor: shape=(10,), dtype=float32, numpy=
  array([0.1985585 , 0.7748644 , 0.27259797, 0.5191392 , 0.04092345,
         0.84705955, 0.57003295, 0.80234504, 0.51251644, 0.07561761],
        dtype=float32)>},
 {'filename': <tf.Tensor: shape=(1,), dtype=string, numpy=
  array([b'/home/lac5440/.cache/pooch/wsi/TCGA-AN-A0G0-